In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC ## **Sample DataFrame Size Measurement**

# COMMAND ----------

# DBTITLE 1,1. Create Sample DataFrame
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import random

# Create sample data
data = [
    (i, f"product_{i}", random.choice(["Electronics", "Clothing", "Food"]), float(random.randint(10, 1000)), random.randint(1, 100))
    for i in range(100000)  # 100K rows
]

# Define schema
schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("quantity", IntegerType(), True)
])

# Create DataFrame
df = spark.createDataFrame(data, schema)
display(df.limit(5))

# COMMAND ----------

# DBTITLE 1,2. Method 1: Column-wise Size Calculation
from pyspark.sql.functions import col, sum, length

size_expr = sum([length(col(c).cast("binary")) for c in df.columns])
estimated_bytes = df.select(size_expr).collect()[0][0]

print(f"Method 1 - Estimated size: {estimated_bytes/(1024**2):.2f} MB")

# COMMAND ----------

# DBTITLE 1,3. Method 2: Write & Measure (Most Accurate)
temp_path = "/tmp/size_measurement"

# Write with current partitioning
df.write.mode("overwrite").parquet(temp_path)

# Get exact size (Databricks)
file_size = sum([f.size for f in dbutils.fs.ls(temp_path) if f.path.endswith(".parquet")])
print(f"Method 2 - Exact disk size: {file_size/(1024**2):.2f} MB")

# Cleanup
dbutils.fs.rm(temp_path, recurse=True)

# COMMAND ----------

# DBTITLE 1,4. Method 3: Spark Internal Estimation (Spark 3.0+)
try:
    estimated_bytes = df._jdf.queryExecution().optimizedPlan().stats().sizeInBytes()
    print(f"Method 3 - Spark internal estimate: {estimated_bytes/(1024**2):.2f} MB")
except:
    print("Method 3 requires Spark 3.0+")

# COMMAND ----------

# DBTITLE 1,5. Compare Results
# MAGIC %md
# MAGIC ### **Size Measurement Results**
# MAGIC | Method | Description | Size |
# MAGIC |--------|-------------|------|
# MAGIC | 1 | Column-wise binary cast | {estimated_bytes/(1024**2):.2f} MB |
# MAGIC | 2 | Physical write measurement | {file_size/(1024**2):.2f} MB |
# MAGIC | 3 | Spark internal estimate | {estimated_bytes/(1024**2):.2f} MB if available |

# COMMAND ----------

# DBTITLE 1,6. Additional Checks
print(f"Number of rows: {df.count():,}")
print(f"Number of columns: {len(df.columns)}")
print(f"Number of partitions: {df.rdd.getNumPartitions()}")

In [0]:
print(random.choice(["Srinivas","Menthula"]))

In [0]:
list_one = [1,22,33,44]
print(sum(list_one))